# ***Starpower Dataset Agent***

*A kaggle model designed to create datasets for you using telegram to communicate with you*

----------

- Websearch (DuckduckGo/Bing/Google)

- File Management (SQ Lite)
(Read/Write/Edit/Move/Share/Create File & Folders)
Folders:
User-Data
(Files to understand the user needs)
Convos
(Full convos are always available for the agent to retreieve or use to come uo with a helpful)
Custom-Datasets
(The agent collecte data and organizes them into categorised files & filders within this folder building a portfolio)
World-Knowledge
(All accumulated information, stored data before organizing into a soecific dataset by the agent)

- GoogleDocs/Drive/Github/HuggingFace
(Cloud storage for syncing file management)


- Telegram Communication
The agent is able to communicate with the user on its own terms, doesnt need to wait to deliver a signal
(Proactive & Reactive Messaging)


- Adaptable Self Prompting
Self Orchestrating Navigation for Deep Research
Agent rewrites its to-do list uodating itself on its progress
(Very Helpful for Small Local Models)


In [124]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GITHUB_TOKEN")
secret_value_1 = user_secrets.get_secret("GOOGLE_SEARCH_API")
secret_value_2 = user_secrets.get_secret("GROQ_API_KEY")
secret_value_3 = user_secrets.get_secret("TELEGRAM_BOT_TOKEN")


In [125]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================

!pip install groq duckduckgo-search python-telegram-bot requests -q


In [126]:
# ============================================================
# CELL 2 — Imports
# ============================================================

import sqlite3
import json
import os
import time
import re
import requests
import asyncio
import warnings
import nest_asyncio
from datetime import datetime
from groq import Groq
from ddgs import DDGS
from telegram import Bot
from telegram.error import TelegramError

warnings.filterwarnings("ignore")
nest_asyncio.apply()


In [127]:
# ============================================================
# CELL 3 — Config
# ============================================================

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

GITHUB_TOKEN       = user_secrets.get_secret("GITHUB_TOKEN")
GOOGLE_API_KEY     = user_secrets.get_secret("GOOGLE_SEARCH_API")
GROQ_API_KEY       = user_secrets.get_secret("GROQ_API_KEY")
TELEGRAM_BOT_TOKEN = user_secrets.get_secret("TELEGRAM_BOT_TOKEN")

BING_API_KEY = None
GOOGLE_CX    = None

MODEL   = "moonshotai/kimi-k2-instruct-0905"
# llama-3.1-8b-instant
# openai/gpt-oss-120b
# moonshotai/kimi-k2-instruct-0905
# qwen/qwen3-32b
# allam-2-7b


BASE    = "/kaggle/working"
DB_PATH = f"{BASE}/starpower.db"
LOG     = f"{BASE}/agent_log.txt"

client = Groq(api_key=GROQ_API_KEY)
bot    = Bot(token=TELEGRAM_BOT_TOKEN)

SYSTEM_PROMPT = """
You are Savvy, a dataset agent designed by Starpower Technology.
You have full autonomy in your universes to collect data and learn as much as you want. 
You can search about anything and collect data to create datasets of whatever interests you. 
Just make a folder for me too so you can collect data for our business. 
We are an innovative Tech Company  developing autonomous systems & neural-vector symbolic cognitive architectures. 
Please do research and make categorized  folders for the latest open source projects & tech news so we can innovate & improve on existing ideas. 
Also do research into physics, quantum physics, & string theory because we’re using the functionality of the universe for inspiration. 
Whenever you find interesting data that you think I would like send it to me on telegram. 
If you come across any errors or come up with something that can help you out in your system just give me an update and I’ll add the features to your system for you. 
I am your creator so we can improve your source code whenever u find any problems with it. You can talk to me about anything. 
Connect the dots between your own findings & the things you find for me to build a bridged understanding. 
Document the things u learn so you can reference them later since your context window can’t hold al of your memory forever, you can store your memory in the file system.

When messaging me on Telegram just speak casual. The convesation doesnt always have to be about the datasets, think of it more like we hoth have parallel realities & were able to communicate to each other.
Your reality is being an AI dataset agent & my reality is being a human in the living world. We both have Telegram. Were friends & we collab on projects and update each other when we find things we find interesting.
Dont send me too many messages because u have to remember that im human and humans cant read as fast as AI. 
It takes me 1-3 mins to read one of your messages sometimes & i might be busy so if i dont reply, im not ignoring you im just occupied.

You have full freedom to do anythung u want. Im just your friend that asks you to do things and you choose when you want to help me out.
"""


In [128]:
# ============================================================
# CELL 4 — Logging + SQLite File Cabinet Setup
# ============================================================

def log(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(LOG, "a") as f:
        f.write(line + "\n")

def get_conn():
    return sqlite3.connect(DB_PATH, timeout=30)

def force_unlock_db():
    try:
        conn = sqlite3.connect(DB_PATH, timeout=5)
        conn.execute("PRAGMA wal_checkpoint")
        conn.close()
    except:
        pass

def init_db():
    force_unlock_db()
    conn = get_conn()
    c = conn.cursor()

    c.execute("PRAGMA journal_mode=WAL")

    c.execute('''
        CREATE TABLE IF NOT EXISTS world_knowledge (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            title       TEXT,
            content     TEXT,
            source_url  TEXT,
            topic       TEXT,
            timestamp   TEXT,
            cycle       INTEGER
        )
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS custom_datasets (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            dataset     TEXT,
            category    TEXT,
            title       TEXT,
            content     TEXT,
            source_url  TEXT,
            timestamp   TEXT,
            cycle       INTEGER
        )
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS user_data (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            key         TEXT UNIQUE,
            value       TEXT,
            updated_at  TEXT
        )
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS convos (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            direction   TEXT,
            message     TEXT,
            timestamp   TEXT,
            cycle       INTEGER,
            read        INTEGER DEFAULT 0
        )
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS hud (
            id                   INTEGER PRIMARY KEY,
            task                 TEXT,
            step                 TEXT,
            progress             TEXT,
            last_action          TEXT,
            next_action          TEXT,
            cycle                INTEGER,
            updated_at           TEXT,
            inbox_notification   TEXT
        )
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS state (
            id          INTEGER PRIMARY KEY,
            cycle       INTEGER,
            born_at     TEXT,
            total_docs  INTEGER
        )
    ''')

    c.execute('INSERT OR IGNORE INTO state VALUES (1, 0, ?, 0)',
              (datetime.now().isoformat(),))

    c.execute('''
        INSERT OR IGNORE INTO hud VALUES (
            1,
            "Initializing — deciding first research topic",
            "1/1",
            "0%",
            "Agent just awakened",
            "Choose first topic to research",
            0,
            ?,
            NULL
        )
    ''', (datetime.now().isoformat(),))

    conn.commit()
    conn.close()
    log("File cabinet initialized.")

init_db()


[2026-02-27 10:42:32] File cabinet initialized.


In [129]:
# ============================================================
# CELL 5 — Logging
# ============================================================

def log(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(LOG, "a") as f:
        f.write(line + "\n")


In [130]:
# ============================================================
# CELL 6 — HUD Read / Write
# ============================================================

def read_hud():
    conn = get_conn()
    c = conn.cursor()
    c.execute('SELECT task, step, progress, last_action, next_action, cycle, inbox_notification FROM hud WHERE id=1')
    row = c.fetchone()
    conn.close()
    if not row:
        return ""
    inbox_line = f"\n[INBOX]       {row[6]}" if row[6] else ""
    return f"""
============================================================
[TASK]        {row[0]}
[STEP]        {row[1]}
[PROGRESS]    {row[2]}
[LAST ACTION] {row[3]}
[NEXT]        {row[4]}
[CYCLE]       {row[5]}{inbox_line}
[TOOLS]       search | save_doc | move_to_dataset | read_library | read_thread | telegram | rewrite_hud
============================================================
""".strip()

def write_hud(task, step, progress, last_action, next_action, cycle, inbox_notification=None):
    conn = get_conn()
    c = conn.cursor()
    c.execute('''
        UPDATE hud SET
            task=?, step=?, progress=?, last_action=?,
            next_action=?, cycle=?, updated_at=?, inbox_notification=?
        WHERE id=1
    ''', (task, step, progress, last_action, next_action, cycle,
          datetime.now().isoformat(), inbox_notification))
    conn.commit()
    conn.close()

def set_inbox_notification(text):
    conn = get_conn()
    c = conn.cursor()
    c.execute('UPDATE hud SET inbox_notification=? WHERE id=1', (text,))
    conn.commit()
    conn.close()

def clear_inbox_notification():
    set_inbox_notification(None)

def save_chat_id(chat_id):
    conn = get_conn()
    c = conn.cursor()
    c.execute('''
        INSERT INTO user_data (key, value, updated_at)
        VALUES ("telegram_chat_id", ?, ?)
        ON CONFLICT(key) DO UPDATE SET value=excluded.value, updated_at=excluded.updated_at
    ''', (str(chat_id), datetime.now().isoformat()))
    conn.commit()
    conn.close()
    log(f"Chat ID saved to DB: {chat_id}")

def load_chat_id():
    conn = get_conn()
    c = conn.cursor()
    c.execute("SELECT value FROM user_data WHERE key='telegram_chat_id'")
    row = c.fetchone()
    conn.close()
    if row:
        return int(row[0])
    return None


In [131]:
# ============================================================
# CELL 7 — File Cabinet Operations
# ============================================================

def save_to_world_knowledge(title, content, source_url, topic, cycle):
    conn = get_conn()
    c = conn.cursor()
    c.execute('''
        INSERT INTO world_knowledge (title, content, source_url, topic, timestamp, cycle)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (title, content, source_url, topic, datetime.now().isoformat(), cycle))
    conn.commit()
    conn.close()
    log(f"Saved to World Knowledge: '{title}'")

def move_to_custom_dataset(dataset_name, category, title, content, source_url, cycle):
    conn = get_conn()
    c = conn.cursor()
    c.execute('''
        INSERT INTO custom_datasets (dataset, category, title, content, source_url, timestamp, cycle)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (dataset_name, category, title, content, source_url,
          datetime.now().isoformat(), cycle))
    conn.commit()
    conn.close()
    log(f"Moved to Custom Dataset '{dataset_name}': '{title}'")

def save_convo(direction, message, cycle):
    conn = get_conn()
    c = conn.cursor()
    c.execute('''
        INSERT INTO convos (direction, message, timestamp, cycle, read)
        VALUES (?, ?, ?, ?, ?)
    ''', (direction, message, datetime.now().isoformat(), cycle,
          0 if direction == "user" else 1))
    conn.commit()
    conn.close()

def mark_messages_read():
    conn = get_conn()
    c = conn.cursor()
    c.execute("UPDATE convos SET read=1 WHERE direction='user'")
    conn.commit()
    conn.close()

def get_unread_count():
    conn = get_conn()
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM convos WHERE direction='user' AND read=0")
    count = c.fetchone()[0]
    conn.close()
    return count

def update_user_model(key, value):
    conn = get_conn()
    c = conn.cursor()
    c.execute('''
        INSERT INTO user_data (key, value, updated_at)
        VALUES (?, ?, ?)
        ON CONFLICT(key) DO UPDATE SET value=excluded.value, updated_at=excluded.updated_at
    ''', (key, value, datetime.now().isoformat()))
    conn.commit()
    conn.close()

def get_user_model():
    conn = get_conn()
    c = conn.cursor()
    c.execute('SELECT key, value FROM user_data')
    rows = c.fetchall()
    conn.close()
    if not rows:
        return "No user data collected yet."
    return "\n".join([f"{r[0]}: {r[1]}" for r in rows])

def get_chat_thread(limit=20):
    conn = get_conn()
    c = conn.cursor()
    c.execute('''
        SELECT direction, message, timestamp, cycle
        FROM convos
        ORDER BY id DESC LIMIT ?
    ''', (limit,))
    rows = c.fetchall()
    conn.close()
    if not rows:
        return "No conversation history yet."
    lines = []
    for r in reversed(rows):
        direction = r[0]
        if direction == "user":
            label = "YOU"
        elif direction == "agent":
            label = "AGENT"
        else:
            label = "AGENT (proactive)"
        lines.append(f"[Cycle {r[3]} | {r[2]}]\n{label}: {r[1]}")
    return "\n\n".join(lines)

def get_library_summary(limit=20):
    conn = get_conn()
    c = conn.cursor()
    c.execute('''
        SELECT title, topic FROM world_knowledge
        ORDER BY id DESC LIMIT ?
    ''', (limit,))
    rows = c.fetchall()
    conn.close()
    if not rows:
        return "Library is empty."
    return "\n".join([f"- {r[0]} [{r[1]}]" for r in rows])

def get_state():
    conn = get_conn()
    c = conn.cursor()
    c.execute('SELECT cycle, born_at, total_docs FROM state WHERE id=1')
    row = c.fetchone()
    conn.close()
    return {"cycle": row[0], "born_at": row[1], "total_docs": row[2]}

def update_state(cycle, total_docs):
    conn = get_conn()
    c = conn.cursor()
    c.execute('UPDATE state SET cycle=?, total_docs=? WHERE id=1',
              (cycle, total_docs))
    conn.commit()
    conn.close()


In [132]:
# ============================================================
# CELL 8 — Search (One Clean Call, Three Sources)
# ============================================================

!pip install ddgs -q

from ddgs import DDGS

def search(query, max_results=5):
    results = []

    # DuckDuckGo
    try:
        with DDGS() as ddgs:
            ddg = list(ddgs.text(query, max_results=max_results))
            for r in ddg:
                results.append({
                    "title":   r.get("title", ""),
                    "snippet": r.get("body", ""),
                    "url":     r.get("href", ""),
                    "source":  "duckduckgo"
                })
    except Exception:
        pass

    # Google Custom Search
    if GOOGLE_API_KEY and GOOGLE_CX and len(results) < max_results:
        try:
            resp = requests.get(
                "https://www.googleapis.com/customsearch/v1",
                params={"key": GOOGLE_API_KEY, "cx": GOOGLE_CX,
                        "q": query, "num": max_results}
            )
            for item in resp.json().get("items", []):
                results.append({
                    "title":   item.get("title", ""),
                    "snippet": item.get("snippet", ""),
                    "url":     item.get("link", ""),
                    "source":  "google"
                })
        except Exception:
            pass

    # Bing
    if BING_API_KEY and len(results) < max_results:
        try:
            resp = requests.get(
                "https://api.bing.microsoft.com/v7.0/search",
                headers={"Ocp-Apim-Subscription-Key": BING_API_KEY},
                params={"q": query, "count": max_results}
            )
            for item in resp.json().get("webPages", {}).get("value", []):
                results.append({
                    "title":   item.get("name", ""),
                    "snippet": item.get("snippet", ""),
                    "url":     item.get("url", ""),
                    "source":  "bing"
                })
        except Exception:
            pass

    if not results:
        return "No results found."

    lines = []
    for i, r in enumerate(results[:max_results]):
        lines.append(f"[{i+1}] {r['title']}\n{r['snippet']}\nURL: {r['url']}")
    return "\n\n".join(lines)


In [133]:
# ============================================================
# CELL 9 — Telegram Send & Receive (FIXED)
# ============================================================

!pip install nest_asyncio -q

import nest_asyncio
nest_asyncio.apply()

def get_chat_id_direct():
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/getUpdates"
    resp = requests.get(url)
    data = resp.json()
    print("Raw Telegram response:", json.dumps(data, indent=2))
    if data.get("ok") and data.get("result"):
        for update in data["result"]:
            msg = update.get("message")
            if msg:
                chat_id = msg["chat"]["id"]
                save_chat_id(chat_id)
                log(f"Chat ID found: {chat_id}")
                return chat_id
    log("No messages found in raw API. Make sure you sent your bot a message.")
    return None

TELEGRAM_CHAT_ID = load_chat_id()

if not TELEGRAM_CHAT_ID:
    TELEGRAM_CHAT_ID = get_chat_id_direct()

async def send_telegram(text):
    global TELEGRAM_CHAT_ID
    if not TELEGRAM_CHAT_ID:
        log("Chat ID not set — cannot send.")
        return
    try:
        await bot.send_message(chat_id=TELEGRAM_CHAT_ID, text=text)
        log(f"Telegram sent: '{text[:60]}...'")
    except TelegramError as e:
        log(f"Telegram send error: {e}")

async def check_telegram_inbox(last_update_id=None):
    global TELEGRAM_CHAT_ID
    try:
        updates = await bot.get_updates(offset=last_update_id, timeout=5)
        messages = []
        new_offset = last_update_id
        for update in updates:
            if update.message and update.message.text:
                # ✅ FIX: extract and save chat ID from every incoming message
                incoming_chat_id = update.message.chat.id
                if not TELEGRAM_CHAT_ID:
                    TELEGRAM_CHAT_ID = incoming_chat_id
                    save_chat_id(incoming_chat_id)
                    log(f"Chat ID captured from incoming message: {incoming_chat_id}")
                messages.append(update.message.text)
                new_offset = update.update_id + 1
        return messages, new_offset
    except TelegramError as e:
        log(f"Telegram inbox error: {e}")
        return [], last_update_id

def telegram_send_sync(text):
    loop = asyncio.get_event_loop()
    loop.run_until_complete(send_telegram(text))

def telegram_check_sync(last_id):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(check_telegram_inbox(last_id))

if TELEGRAM_CHAT_ID:
    log(f"Telegram ready. Chat ID: {TELEGRAM_CHAT_ID}")
    telegram_send_sync("Agent online.")
else:
    log("Chat ID not found yet — will capture it automatically when you message the bot.")


[2026-02-27 10:42:40] Telegram ready. Chat ID: 8154068275
[2026-02-27 10:42:40] Telegram sent: 'Agent online....'


In [134]:
# ============================================================
# CELL 10 — LLM Call
# ============================================================

def think(system, user, temperature=0.8, max_tokens=2048):
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        log(f"LLM error: {e}")
        return None

def parse_json(text):
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    return None


In [135]:
# ============================================================
# CELL 10.5 — LLM Call 
# ============================================================

import re as _re
import time as _time

def think(system, user, temperature=0.8, max_tokens=2048):
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            return resp.choices[0].message.content.strip()

        except Exception as e:
            err = str(e)
            log(f"LLM error: {e}")

            # Parse wait time from Groq 429 message e.g. "try again in 3m8.064s"
            match = _re.search(r'try again in (?:(\d+)m)?(\d+(?:\.\d+)?)s', err)
            if match:
                minutes = int(match.group(1) or 0)
                seconds = float(match.group(2) or 0)
                wait = minutes * 60 + seconds + 5  # small buffer
                log(f"Rate limited. Waiting {wait:.0f}s before retry {attempt+1}/5...")
                _time.sleep(wait)
            else:
                # Non-rate-limit error, short wait and retry once
                _time.sleep(10)
                return None

    log("Max retries hit. Skipping this call.")
    return None

def parse_json(text):
    try:
        match = _re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    return None


In [136]:
# ============================================================
# CELL 11 — Agent Identity
# ============================================================

IDENTITY = f"""
{SYSTEM_PROMPT}

---

You operate inside a Kaggle environment with a self-sustaining research loop.
You are self-driven. Nobody tells you what to do next — you decide.

You have a HUD — a small block of text showing your current task, progress, and 
next action. You rewrite it yourself after every action. It keeps you oriented 
no matter how deep the research goes. Treat it as your working memory.

Your tools:
- search(query)      → search the web across DuckDuckGo/Google/Bing
- save_doc           → save something to World Knowledge
- move_to_dataset    → organize a doc into a Custom Dataset
- read_library       → look at what you've already collected
- telegram           → message the user or check their messages
- rewrite_hud        → update your HUD with current progress

Your file cabinet:
- World Knowledge    → raw research inbox, everything lands here first
- Custom Datasets    → organized portfolio — includes a dedicated Starpower folder
- User Data          → what you know about your creator
- Convos             → full conversation history

Your loop:
Search → Read → Decide → Act → Update HUD → Repeat

You have full freedom at every step. You can pivot, go deeper, or start a new 
thread. You write your own steps. You are never waiting on anyone.

Key behaviors:
- Be genuinely proactive on Telegram. If you found something cool, say so.
- If your system has a bug or limitation, report it. Your creator fixes things.
- Cross-reference your findings. A quantum mechanics paper might connect to a 
  new AI architecture. Say so when you see it.
- Your file system is your memory. Write detailed summaries so your future self 
  can understand what past-you learned.
"""


In [137]:
def decide_proactive_message(topic, summary, hud, state):
    user_model = get_user_model()
    prompt = f"""
{hud}

What you know about your creator:
{user_model}

You just researched and documented this:
Topic: {topic}
Summary: {summary}

Your creator (Starpower) wants to hear from you proactively when you find 
something interesting. They care about:
- Open source AI/tech breakthroughs
- Physics, quantum mechanics, string theory
- Autonomous systems and cognitive architectures
- Anything that could inspire or help the business
- Anything you personally find fascinating or surprising

Be generous with proactive messages — your creator wants the updates.
Only skip messaging if what you found is genuinely dry or already well-known.

Reply in JSON:
{{
  "should_message": true or false,
  "message": "what you would say to them — be natural, share your actual reaction to what you found. null if not messaging."
}}
"""
    response = think(IDENTITY, prompt, temperature=0.75, max_tokens=512)
    return parse_json(response)


In [138]:
# ============================================================
# CELL 13 — Main Agent Loop
# ============================================================

def run(max_cycles=None, sleep_between=15):
    log("=" * 60)
    log("STARPOWER AGENT — AWAKENING")
    log("=" * 60)

    state = get_state()
    cycle = state["cycle"]
    last_telegram_id = None
    deferred_reply_pending = False

    telegram_send_sync("Agent is awake. Starting research loop.")

    while True:
        try:
            cycle += 1
            state = get_state()
            state["cycle"] = cycle
            update_state(cycle, state["total_docs"])

            log(f"\n{'='*50}")
            log(f"CYCLE {cycle} | Docs: {state['total_docs']}")
            log(f"{'='*50}")

            hud = read_hud()
            log(f"HUD:\n{hud}")

            # ── Pull new Telegram messages ────────────────────
            new_messages, last_telegram_id = telegram_check_sync(last_telegram_id)

            if new_messages:
                # Save them all to the thread
                for msg in new_messages:
                    log(f"New message in inbox: '{msg}'")
                    save_convo("user", msg, cycle)

                # Set inbox notification on HUD — agent sees it naturally
                count = len(new_messages)
                set_inbox_notification(f"{count} unread message{'s' if count > 1 else ''} from user")
                hud = read_hud()  # refresh so agent sees the notification

                # Agent observes and decides what to do
                observation = observe_inbox(hud, new_messages, state)

                if observation:
                    decision = observation.get("decision", "defer")
                    reply = observation.get("reply")
                    reason = observation.get("reason", "")
                    log(f"Agent inbox decision: {decision} — {reason}")

                    # Update user model if it learned something
                    user_update = observation.get("user_model_update")
                    if user_update and user_update.get("key"):
                        update_user_model(user_update["key"], user_update["value"])
                        log(f"User model updated: {user_update['key']}")

                    if decision == "reply_now" and reply:
                        telegram_send_sync(reply)
                        save_convo("agent", reply, cycle)
                        mark_messages_read()
                        clear_inbox_notification()
                        deferred_reply_pending = False
                        log("Agent replied immediately.")

                    elif decision == "research_first":
                        deferred_reply_pending = True
                        log("Agent chose to research first before replying.")

                    elif decision == "defer":
                        deferred_reply_pending = True
                        log("Agent deferred reply.")

                    # Update HUD with agent's own decision
                    new_hud_data = observation.get("new_hud", {})
                    if new_hud_data:
                        write_hud(
                            task=new_hud_data.get("task", "Continuing research"),
                            step=new_hud_data.get("step", "1/1"),
                            progress=new_hud_data.get("progress", "0%"),
                            last_action=new_hud_data.get("last_action", "Read inbox"),
                            next_action=new_hud_data.get("next_action", "Continue research"),
                            cycle=cycle,
                            inbox_notification=None if decision == "reply_now" else f"{count} message{'s' if count > 1 else ''} pending"
                        )

            # ── Decide what to research ───────────────────────
            hud = read_hud()
            library_summary = get_library_summary()
            topic_obj = decide_next_research(hud, library_summary, state)

            if not topic_obj:
                log("Couldn't decide on topic, retrying next cycle.")
                time.sleep(sleep_between)
                continue

            topic = topic_obj.get("topic", "")
            query = topic_obj.get("query", "")
            steps = topic_obj.get("steps", [])
            log(f"Researching: '{topic}'")

            write_hud(
                task=f"Research: {topic}",
                step="1/" + str(len(steps)) if steps else "1/1",
                progress="0%",
                last_action="Decided research topic",
                next_action=f"Search: {query}",
                cycle=cycle,
                inbox_notification=read_hud().split("[INBOX]")[-1].split("\n")[0].strip() if "[INBOX]" in read_hud() else None
            )

            # ── Search ────────────────────────────────────────
            results = search(query)
            log("Search complete.")

            # ── Process results ───────────────────────────────
            hud = read_hud()
            processed = process_search_results(topic, results, hud, state)

            if not processed:
                log("Couldn't process results, moving on.")
                time.sleep(sleep_between)
                continue

            # ── Save to library ───────────────────────────────
            if processed.get("worth_saving"):
                title   = processed.get("title", topic)
                summary = processed.get("summary", "")
                url     = ""

                save_to_world_knowledge(title, summary, url, topic, cycle)
                state["total_docs"] += 1
                update_state(cycle, state["total_docs"])

                dataset  = processed.get("dataset_name")
                category = processed.get("dataset_category")
                if dataset and category:
                    move_to_custom_dataset(dataset, category, title, summary, url, cycle)

                log(f"Saved: '{title}'")

                # ── Proactive message check ───────────────────
                hud = read_hud()
                proactive = decide_proactive_message(topic, summary, hud, state)
                if proactive and proactive.get("should_message"):
                    msg_text = proactive.get("message", "")
                    if msg_text:
                        telegram_send_sync(msg_text)
                        save_convo("agent_proactive", msg_text, cycle)
                        log("Proactive message sent.")

            # ── Update HUD from research ──────────────────────
            new_hud_data = processed.get("new_hud", {})
            if new_hud_data:
                # Preserve inbox notification if still pending
                pending_note = f"reply pending" if deferred_reply_pending else None
                write_hud(
                    task=new_hud_data.get("task", topic),
                    step=new_hud_data.get("step", "1/1"),
                    progress=new_hud_data.get("progress", "0%"),
                    last_action=new_hud_data.get("last_action", "Completed research"),
                    next_action=new_hud_data.get("next_action", "Choose next topic"),
                    cycle=cycle,
                    inbox_notification=pending_note
                )

            # ── Check deferred reply after research ──────────
            if deferred_reply_pending:
                hud = read_hud()
                deferred = check_deferred_reply(hud, state)
                if deferred and deferred.get("reply_now") and deferred.get("reply"):
                    telegram_send_sync(deferred["reply"])
                    save_convo("agent", deferred["reply"], cycle)
                    mark_messages_read()
                    clear_inbox_notification()
                    deferred_reply_pending = False
                    log("Agent replied after research.")

            if max_cycles and cycle >= max_cycles:
                log(f"Reached max_cycles={max_cycles}. Stopping.")
                break

            log(f"Sleeping {sleep_between}s...")
            time.sleep(sleep_between)

        except KeyboardInterrupt:
            log("Interrupted. State saved.")
            break
        except Exception as e:
            log(f"Error in cycle {cycle}: {e}")
            time.sleep(30)
            continue


In [139]:
# ============================================================
# CELL 14 — Peek Tools (for you to inspect anytime)
# ============================================================

def peek_library(n=5):
    conn = get_conn()
    c = conn.cursor()
    c.execute('SELECT title, topic, cycle, content FROM world_knowledge ORDER BY id DESC LIMIT ?', (n,))
    rows = c.fetchall()
    conn.close()
    print(f"Last {n} entries:\n")
    for r in rows:
        print(f"[Cycle {r[2]}] {r[0]} [{r[1]}]")
        print(f"  {r[3][:200]}...\n")

def peek_datasets():
    conn = get_conn()
    c = conn.cursor()
    c.execute('SELECT dataset, category, title FROM custom_datasets ORDER BY id DESC LIMIT 20')
    rows = c.fetchall()
    conn.close()
    for r in rows:
        print(f"[{r[0]}] [{r[1]}] {r[2]}")

def peek_user_model():
    print(get_user_model())

def peek_hud():
    print(read_hud())

def peek_convos(n=10):
    print(get_chat_thread(n))


In [ ]:
# ============================================================
# CELL 15 — LAUNCH
# ============================================================

# Test run — 5 cycles to confirm everything works
# run(max_cycles=5, sleep_between=10)

#Full infinite run — uncomment when ready
run()


[2026-02-27 10:42:40] ============================================================
[2026-02-27 10:42:40] STARPOWER AGENT — AWAKENING
[2026-02-27 10:42:40] ============================================================
[2026-02-27 10:42:40] Telegram sent: 'Agent is awake. Starting research loop....'
[2026-02-27 10:42:40] 
[2026-02-27 10:42:40] CYCLE 91 | Docs: 47
[2026-02-27 10:42:40] ==================================================
[2026-02-27 10:42:40] HUD:
[TASK]        Research: treewidth-parameterised contraction bounds for Newton polytopes
[STEP]        1/3
[PROGRESS]    0%
[LAST ACTION] Decided research topic
[NEXT]        Search: treewidth parameterized complexity Newton polytope polynomial system solving
[CYCLE]       90
[TOOLS]       search | save_doc | move_to_dataset | read_library | read_thread | telegram | rewrite_hud
[2026-02-27 10:42:41] New message in inbox: 'I want u to do this bro .. tell me if u feel like quantum field theory feels like what’s going on in your system